# WP37 — Online ELO & Performance Registry (v0.4)
## GlickoRating · ELORegistry · DomainRegistry · ZPDRatingBridge

Demonstrates **WP37**: a persistent, uncertainty-aware Glicko-2 rating system that tracks agent improvement across sessions and domains.

Runtime: **~2 min** (no GPU, SQLite only)

In [ ]:
import sys, os, importlib
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('Prometheus_v0_PoC'):
        os.system('git clone -b wp16-notebook-only https://github.com/pmcray/Prometheus_v0_PoC.git')
    os.chdir('Prometheus_v0_PoC'); sys.path.insert(0, '/content/Prometheus_v0_PoC')
    importlib.invalidate_caches()
else:
    repo_root = os.path.abspath(os.path.join(os.getcwd(), '..')); 
    if repo_root not in sys.path: sys.path.insert(0, repo_root)
    importlib.invalidate_caches()
import warnings; warnings.filterwarnings('ignore')
import time, random, numpy as np, matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({'figure.figsize': (14, 7), 'font.size': 11})
SEED = 42; random.seed(SEED); np.random.seed(SEED)
import prometheus; print(f'Prometheus {prometheus.__version__} OK')

In [ ]:
from prometheus.wp37_elo_registry import (
    GlickoRating, ELORegistry, DomainRegistry, ZPDRatingBridge,
    glicko2_update, verify_wp37_exit_criteria
)
print('WP37 imports OK')
print('Glicko-2 defaults: mu=1500  phi=350  sigma=0.06')

In [ ]:
# ELORegistry: register agents and record results
registry = ELORegistry(':memory:', domain='synthesis')
agents = ['prometheus_v0', 'prometheus_v1', 'prometheus_v2', 'static_baseline']
for a in agents:
    r = registry.register_agent(a)
    print(f'  Registered: {a:20s}  mu={r.mu:.0f}  phi={r.phi:.0f}')

In [ ]:
# Simulate 20 games: prometheus agents beat static_baseline
import random
random.seed(42)
for game in range(20):
    a = random.choice(['prometheus_v0','prometheus_v1','prometheus_v2'])
    b = 'static_baseline'
    # Prometheus wins 75% of the time
    outcome = 1.0 if random.random() < 0.75 else 0.0
    entry = registry.record_result(a, b, outcome)

# Also record some inter-prometheus games
for game in range(10):
    a, b = random.sample(['prometheus_v0','prometheus_v1','prometheus_v2'], 2)
    entry = registry.record_result(a, b, random.choice([1.0, 0.5, 0.0]))

registry.update_ratings()
print('After 30 games:')
print(f'  {"Agent":<22} {"Rating (mu)":>12} {"Uncertainty (phi)":>18}')
print('  ' + '-'*55)
for agent_id, mu, phi in registry.leaderboard():
    print(f'  {agent_id:<22} {mu:>12.1f} {phi:>18.1f}')

In [ ]:
# Rating history tracking
history = registry.rating_history('prometheus_v0')
print(f'prometheus_v0 rating history: {len(history)} entries')
for ts, mu in history[:5]:
    import datetime
    t_str = datetime.datetime.fromtimestamp(ts).strftime('%H:%M:%S')
    print(f'  {t_str}  mu={mu:.1f}')

In [ ]:
# DomainRegistry — multi-domain ratings
dom = DomainRegistry(':memory:', ['chess','go','theorems','synthesis'])
for domain in ['chess','go','theorems']:
    dom.record_result('agent_A', 'agent_B', 1.0, domain)
    dom.record_result('agent_A', 'agent_C', 0.5, domain)
    dom.record_result('agent_B', 'agent_C', 0.0, domain)
dom.update_all()
print('Multi-domain leaderboards:')
for domain in ['chess','go','theorems']:
    board = dom.leaderboard(domain, top_k=3)
    print(f'  [{domain}]:', ' | '.join(f'{a}:{mu:.0f}' for a,mu,_ in board))

In [ ]:
# ZPDRatingBridge — convert rating to curriculum difficulty
bridge = ZPDRatingBridge(base_mu=1200, range_mu=600, zpd_lo_base=0.40, zpd_lo_max=0.70)
test_ratings = [1100, 1200, 1400, 1600, 1800]
print('ZPD difficulty bridge:')
print(f'  {"Rating":>8} {"Multiplier":>12} {"ZPD-lo":>8}')
for mu in test_ratings:
    d = bridge.to_dict(mu)
    print(f'  {mu:>8.0f} {d["multiplier"]:>12.3f} {d["effective_zpd_lo"]:>8.3f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel A: leaderboard bar chart
ax = axes[0]
board = registry.leaderboard()
agent_names = [a for a,_,_ in board]
ratings     = [mu for _,mu,_ in board]
phis        = [phi for _,_,phi in board]
bars = ax.barh(agent_names, ratings, color=['#4CAF50' if 'prometheus' in a else '#F44336' for a in agent_names],
               edgecolor='black', alpha=0.85)
ax.errorbar(ratings, range(len(board)), xerr=phis, fmt='none', color='black', capsize=5)
ax.set_xlabel('Glicko-2 Rating (mu)'); ax.set_title('Leaderboard with Uncertainty Bars', fontweight='bold')
ax.axvline(1500, color='gray', linestyle='--', alpha=0.5, label='Default (1500)')
ax.legend()

# Panel B: ZPD bridge curve
ax2 = axes[1]
mus  = list(range(900, 2000, 10))
zpds = [bridge.effective_zpd_lo(m) for m in mus]
mults= [bridge.multiplier(m) for m in mus]
ax2.plot(mus, zpds,  'b-', lw=2, label='ZPD lower bound')
ax2.plot(mus, mults, 'r--', lw=2, label='Multiplier')
ax2.set_xlabel('Rating (mu)'); ax2.set_ylabel('Value')
ax2.set_title('ZPDRatingBridge: Rating -> Curriculum Difficulty', fontweight='bold'); ax2.legend()
ax2.axvline(1500, color='gray', linestyle=':', alpha=0.4)

# Panel C: per-domain rating bar
ax3 = axes[2]
domains = ['chess','go','theorems']
x = range(3)
for i, (domain, clr) in enumerate(zip(domains, ['#2196F3','#FF9800','#9C27B0'])):
    board3 = dom.leaderboard(domain, top_k=1)
    if board3:
        ax3.bar(i, board3[0][1], color=clr, edgecolor='black', alpha=0.85, label=domain)
ax3.set_xticks(list(x)); ax3.set_xticklabels(domains)
ax3.set_ylabel('Top Agent Rating (mu)'); ax3.set_title('Top Rating per Domain', fontweight='bold')
ax3.legend()

fig.suptitle('WP37: Online ELO & Performance Registry', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('wp37_elo_registry.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved wp37_elo_registry.png')

In [ ]:
criteria = verify_wp37_exit_criteria(registry, dom, bridge)
print('WP37 Exit Criteria Verification'); print('='*60)
for c, ok in criteria.items():
    print(f'  {"PASS" if ok else "FAIL"}  {c}')
if all(criteria.values()): print('\nAll WP37 exit criteria satisfied.')

---
## Conclusions

**WP37** provides persistent, calibrated agent ratings:
- Glicko-2 models *uncertainty* (phi) — new agents update faster
- `ELORegistry` persists to SQLite; survives session restarts
- `ZPDRatingBridge` feeds rating back into WP31 curriculum difficulty

### References
- Elo (1978) *The Rating of Chess Players, Past and Present*
- Glickman (1995, 2012) Glicko / Glicko-2
- Silver et al. (2016) *AlphaGo* self-play rating tracking